# Tiny Dreamer Highway — H100 AMP Run

**Name:** Esteban  
**Course:** CSC 580 AI 2  
**Assignment:** Final Project — Dream the Road  
**AI tools consulted:** GitHub Copilot

This notebook pushes H100 utilisation to the maximum with:
- **bf16 Automatic Mixed Precision** – native H100 tensor-core dtype
- **FlashAdamW** fused optimizer (optional; auto-falls back to standard AdamW)
- **batch_size 256**, **16× model/behavior updates per cycle**
- **Aggressive offroad penalty (3.0)** for driving stability

Compare runtime and metrics against the standard H100 notebook (04).

## Setup — mount Drive, clone repo, install package

In [1]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO_URL = 'https://github.com/estmon8u/CSC_580_Final_Project.git'
DRIVE_ROOT = Path('/content/drive/MyDrive/CSC_580_Final_Project')
ARTIFACT_ROOT = DRIVE_ROOT / 'artifacts'

for path in [DRIVE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / 'training_runs']:
    path.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%%bash
set -e
REPO_URL='https://github.com/estmon8u/CSC_580_Final_Project.git'
if [ ! -d /content/CSC_580_Final_Project/.git ]; then
  git clone "${REPO_URL}" /content/CSC_580_Final_Project
else
  cd /content/CSC_580_Final_Project
  git pull --ff-only origin main
fi
cd /content/CSC_580_Final_Project
python -m pip install --upgrade pip --quiet
python -m pip install -e . --quiet
# FlashAdamW — optional; falls back gracefully if install fails
python -m pip install flashoptim --quiet || echo "flashoptim unavailable, using standard AdamW"

Updating 83c0e73..db07143
Fast-forward
 examples/h100_amp_experiment.yaml | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)


From https://github.com/estmon8u/CSC_580_Final_Project
 * branch            main       -> FETCH_HEAD
   83c0e73..db07143  main       -> origin/main


In [3]:
import json
import sys
import torch
from pathlib import Path

PROJECT_ROOT = Path('/content/CSC_580_Final_Project')
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tiny_dreamer_highway.config import load_experiment_config
from tiny_dreamer_highway.training import run_training_experiment

CONFIG_PATH = PROJECT_ROOT / 'examples' / 'h100_amp_experiment.yaml'
config = load_experiment_config(CONFIG_PATH)
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
print('Loaded config from:', CONFIG_PATH)
print('GPU:', gpu_name)
if 'H100' not in gpu_name:
    print('Warning: this notebook is intended for H100-class runtimes.')
print('Batch size:', config.training.batch_size)
print('AMP enabled:', config.training.use_amp, f'({config.training.amp_dtype})')
print('Flash optimizer:', config.training.use_flash_optimizer)
print('World-model updates/cycle:', config.training.world_model_updates_per_cycle)
print('Behavior updates/cycle:', config.training.behavior_updates_per_cycle)

# Quick FlashAdamW availability check
try:
    import flashoptim  # noqa: F401
    print('FlashAdamW: AVAILABLE')
except ImportError:
    print('FlashAdamW: not installed — will use standard AdamW')

Loaded config from: /content/CSC_580_Final_Project/examples/h100_amp_experiment.yaml
GPU: NVIDIA H100 80GB HBM3
Batch size: 256
AMP enabled: True (bfloat16)
Flash optimizer: True
World-model updates/cycle: 16
Behavior updates/cycle: 16
FlashAdamW: AVAILABLE


In [4]:
RUN_NAME = 'h100_amp_run_001'
RUN_ARTIFACT_ROOT = ARTIFACT_ROOT / 'training_runs' / RUN_NAME
RUN_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

CYCLES = None
WARM_START_STEPS = None
POLICY_STEPS = None
CHECKPOINT_INTERVAL = None
RESUME_FROM = None

print('Run name:', RUN_NAME)
print('Effective cycles:', config.training.cycles if CYCLES is None else CYCLES)

Run name: h100_amp_run_001
Effective cycles: 500


In [ ]:
print('Launching H100 AMP run. Per-cycle progress lines will appear below.')

training_summary = run_training_experiment(
    config,
    RUN_ARTIFACT_ROOT,
    cycles=CYCLES,
    warm_start_steps=WARM_START_STEPS,
    policy_steps=POLICY_STEPS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    resume_from=RESUME_FROM,
)

print('Completed cycles:', training_summary.completed_cycles)
print('Latest checkpoint:', training_summary.latest_checkpoint)
print('Latest metrics:', training_summary.latest_record)

Launching H100 AMP run. Per-cycle progress lines will appear below.
[optimizer] cast models to bfloat16 for FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[optimizer] using FlashAdamW
[train] starting run | cycles=500 | start_step=1 | warm_start_steps=2000 | policy_steps=64 | device=cuda


In [ ]:
summary_path = training_summary.log_dir / 'latest_summary.json'
summary_payload = json.loads(summary_path.read_text(encoding='utf-8'))
summary_payload

In [ ]:
from IPython.display import Image, display
from tiny_dreamer_highway.evaluation import export_training_history_artifacts

analysis_outputs = export_training_history_artifacts(
    training_summary.log_dir / 'cycle_metrics.csv',
    RUN_ARTIFACT_ROOT / 'analysis',
    prefix=RUN_NAME,
)

display(Image(filename=str(analysis_outputs['curves'])))
json.loads(analysis_outputs['summary'].read_text(encoding='utf-8'))

## Agent Driving Demo

Record a short GIF showing the trained policy driving in the real highway-env. The checkpoint from the run above is used to load the actor and world model, then the agent is rolled out for a few episodes.

In [ ]:
from IPython.display import Image, display
import importlib
import tiny_dreamer_highway.evaluation as evaluation_pkg

try:
    evaluation_pkg = importlib.reload(evaluation_pkg)
    record_demo_videos = evaluation_pkg.record_demo_videos
except (AttributeError, ImportError):
    from tiny_dreamer_highway.evaluation.policy_rollout import record_demo_videos

print('Using demo recorder from:', record_demo_videos.__module__)

demo_outputs = record_demo_videos(
    config,
    checkpoint_path=training_summary.latest_checkpoint,
    output_dir=RUN_ARTIFACT_ROOT / 'demo_videos',
    num_episodes=2,
    max_steps=1000,
    fps=15,
    seed=config.seed,
    prefix=RUN_NAME,
    device=config.device,
)

for gif_path in demo_outputs.video_paths:
    print(f'\n{gif_path.name}')
    display(Image(filename=str(gif_path)))